*This mathematical model and its corresponding code have been modified by **Jafar Namdar**, Faculty at Eli Broad College of Business, MSU, solely for teaching purposes — specifically to introduce undergraduate Supply Chain Management students to supply chain network design. The original model and notebook were created by the Gurobi team.*

**Modifications include:** parameter adjustments, added/removed constraints, revised objective function, updated widgets, incorporated inventory, shortage, and expanded service level reporting.

**Disclaimer:** This material is intended for educational use only and should not be used for live supply chain operations.

**Original sources:**
- [Gurobi Jupyter model example](https://www.gurobi.com/jupyter_models/supply-network-design/)  
- [Google Colab version](https://colab.research.google.com/github/Gurobi/modeling-examples/blob/master/supply_network_design/supply_network_design_2.ipynb#scrollTo=M9VacUIplKl-)

# 🎮 Network Rescue: The UK Distribution Challenge
### A supply chain network design game for senior SCM students

**Team name:** ______________________ **Mandate card:** ____

| Role | Name | Responsibility |
|---|---|---|
| Model Driver | | Runs the notebook, changes parameters, resets to baseline |
| Analyst | | Records every run in the Scenario Log, checks the math |
| Strategist | | Connects numbers to business decisions and trade-offs |
| Presenter / Skeptic | | Challenges the team's logic, leads the final presentation |

*Teams of 3 → combine Strategist and Presenter.*

## The situation

**Albion Materials Ltd.** makes a bulk product at two UK factories (Liverpool and Brighton) and moves it to six customers, either directly or through a set of depots. The COO has hired your team as consultants. She wants three things:

1. A clear diagnosis of the **current optimal network**.
2. Evidence of how that network behaves under **stress**.
3. A **recommendation** for a strategic mandate the board has handed you, which you will present in class.

You will use the optimization model in this notebook as your analytical engine. Your credibility depends on two things: getting the numbers right, and knowing when the numbers are not enough.

## How the game works

| Round | Type | What you do | Points |
|---|---|---|---|
| 1. Baseline Diagnosis | 🎯 Targeted | Read and interpret the optimal solution | 15 |
| 2. Stress Tests | 🎯 Targeted | Change one parameter at a time and explain the result | 20 |
| 3. Model Audit | 🔍 Critical thinking | Find where the model disagrees with reality | 10 (+3 bonus) |
| 4. Innovation Challenge | 💡 Open-ended | Solve your team's mandate card and go beyond the model | 35 (+3 bonus) |
| 5. Board Presentation | 🎤 Present in class | Pitch your recommendation and survive the Board Twist | 20 |
| | | **Total** | **100 (+6 bonus)** |

The full grading rubric is at the end of this notebook.

## Rules of play

1. **Reset before every what-if.** Re-run the model cell (Part 2) to restore all default values, then change *only* what the question asks. Answers built on stacked changes will not match the key.
2. **Log every run** in the Scenario Log (Part 6). If it is not in the log, it did not happen.
3. **Show your math.** For targeted questions, a number without a calculation or explanation earns partial credit at most.
4. **Board policy locks** (unless your mandate card says otherwise): Total inventory cap stays at **20,000** and shortage cost stays at **$5/ton**.
5. **Alternative optima exist.** If your flows differ from another team's but the total cost is identical, you may both be right. Explain, do not panic.

---
# Part 1 — Briefing Book
Read this before touching the model. It defines the network, the data, and the math.

# 📘 Briefing Book — Supply Network Design 2

## Objective and Prerequisites

Take your supply chain network design skills to the next level in this example. We’ll show you how – given a set of factories, depots, and customers – you can use mathematical optimization to determine which depots to open or close in order to minimize overall costs.

This model is example 20 from the fifth edition of Model Building in Mathematical Programming, by H. Paul Williams on pages 275-276 and 332-333.

This example is of beginning difficulty; we assume that you know Python and have some knowledge of the Gurobi Python API and building mathematical optimization models.


---
## Problem Description

In this problem, we have six end customers, each with a known demand for a product. Customer demand can be satisfied from a set of six depots, or directly from a set of two factories. Each depot can support a maximum volume of product moving through it, and each factory can produce a maximum amount of product. There are known costs associated with transporting the product, from a factory to a depot, from a depot to a customer, or from a factory directly to a customer.  

This extended version also considers:
- Reliability-adjusted capacities for factories and depots  
- Inventory at all tiers with holding costs  
- Shortages with penalties  
- A network-wide total inventory cap  
- Forced-open depot options  
- Expansion choice for Birmingham

Our supply network has two factories, in Liverpool and Brighton, that produce a product. Each has a maximum production capacity:

| Factory | Supply (tons) |
| --- | --- |
| Liverpool | 150,000 |
| Brighton | 200,000 |

The product can be shipped from a factory to a set of six depots. Each depot has a maximum throughput. Depots don't produce or consume the product; they simply pass the product through to customers.

| Depot | Throughput (tons) |
| --- | --- |
| Newcastle | 70,000 |
| Birmingham | 50,000 |
| London | 100,000 |
| Exeter | 40,000 |
| Bristol | 30,000 |
| Northampton | 25,000 |

Opening a depot has a cost:

| Depot | Cost to open |
| --- | --- |
| Newcastle | 10,000 |
| Exeter | 5,000 |
| Bristol | 12,000 |
| Northampton | 4,000 |

We also have the option of expanding the capacity at Birmingham by 20,000 tons, for a cost of \$3000.

Our network has six customers, each with a given demand.

| Customer | Demand (tons) |
| --- | --- |
| C1 | 50,000 |
| C2 | 10,000 |
| C3 | 40,000 |
| C4 | 35,000 |
| C5 | 60,000 |
| C6 | 20,000 |

Shipping costs are given in the following table (in dollars per ton). A '-' in the table indicates that that combination is not possible.

| To | Liverpool | Brighton | Newcastle | Birmingham | London | Exeter | Bristol | Northampton |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **Depots** |||||||||
| Newcastle   | 0.5 | -   |
| Birmingham  | 0.5 | 0.3 |
| London      | 1.0 | 0.5 |
| Exeter      | 0.2 | 0.2 |
| Bristol     | 0.6 | 0.4 |
| Northampton | 0.4 | 0.3 |
| **Customers** |||||||||
| C1 | 1.0 | 2.0 | -   | 1.0 | -   | -   | 1.2 | -   |
| C2 | -   | -   | 1.5 | 0.5 | 1.5 | -   | 0.6 | 0.4 |
| C3 | 1.5 | -   | 0.5 | 0.5 | 2.0 | 0.2 | 0.5 | -   |
| C4 | 2.0 | -   | 1.5 | 1.0 | -   | 1.5 | -   | 0.5 |
| C5 | -   | -   | -   | 0.5 | 0.5 | 0.5 | 0.3 | 0.6 |
| C6 | 1.0 | -   | 1.0 | -   | 1.5 | 1.5 | 0.8 | 0.9 |

The questions to be answered:
1. Which depots should be opened?  
2. Should Birmingham be expanded?  
3. How should product be routed and where should inventory be placed?

---
## Model Formulation

### Sets and Indices

$$
F = \{\text{Liverpool}, \text{Brighton}\} \quad \text{(factories)}
$$

$$
D = \{\text{Newcastle}, \text{Birmingham}, \text{London}, \text{Exeter}, \text{Bristol}, \text{Northampton}\} \quad \text{(depots)}
$$

$$
C = \{\text{C1}, \text{C2}, \text{C3}, \text{C4}, \text{C5}, \text{C6}\} \quad \text{(customers)}
$$

$$
A \subseteq (F \cup D) \times (D \cup C) \quad \text{(transport arcs)}
$$

$$
D_{\text{force}} \subseteq D \quad \text{(forced-open depots)}
$$

| Symbol | Description |
|--------|-------------|
| $$ c_{ij}$$ | Transport cost per unit from node \( i \) to node \( j \) |
| $$ s_f $$ | Nominal supply capacity of factory \( f \) |
| $$ r^F_f $$ | Reliability of factory \( f \) |
| $$ t_d $$ | Nominal throughput of depot \( d \) |
| $$ r^D_d $$ | Reliability of depot \( d \) |
| $$ q_c $$ | Demand of customer \( c \) |
| $$ \pi_c $$ | Shortage penalty per unit at customer \( c \) |
| $$ \alpha_d $$ | Opening cost of depot \( d \) |
| $$ h^F_f, h^D_d, h^C_c $$ | Holding costs per unit at factories, depots, customers |
| $$ \Delta $$ | Extra capacity if Birmingham expands |
| $$ \kappa $$ | Cost of expanding Birmingham |
| $$ M $$ | Maximum number of open depots |
| $$ I_{\max} $$ | Total ending inventory cap |

Effective capacities:
$$
S_f = s_f \cdot r^F_f, \quad T_d = t_d \cdot r^D_d
$$

### Decision Variables
- $$ x_{ij} \ge 0 $$: flow from \( i \) to \( j \)  
- $$y_d \in \{0,1\} $$: depot \( d \) is open  
- $$ z \in \{0,1\} $$: Birmingham expands  
- $$ s_c \ge 0 $$: shortage at customer \( c \)  
-  $$I^F_f, I^D_d, I^C_c \ge 0 $$: ending inventory at factories, depots, customers  

### Objective Function
$$
\begin{aligned}
\min \quad &
\sum_{(i,j) \in A} c_{ij} x_{ij}
+ \sum_{d \in D} \alpha_d y_d
+ \kappa z
+ \sum_{c \in C} \pi_c s_c \\
&+ \sum_{f \in F} h^F_f I^F_f
+ \sum_{d \in D} h^D_d I^D_d
+ \sum_{c \in C} h^C_c I^C_c
\end{aligned}
$$

### Constraints
**Factory capacity with inventory**:- Flow of goods from a factory must respect the sum of maximum capacity and inventory at that location.

$$\sum_{j: (f,j) \in A} x_{fj}  \le S_f +I^F_f\quad \forall f \in F$$


**Depot flow balance with inventory**: Whatever flows into depot must flow out ( depot cannot flow 1000 if it only receives 10)
$$
\sum_{i: (i,d) \in A} x_{id} = \sum_{j: (d,j) \in A} x_{dj} \quad \forall d \in D
$$

**Customer demand with shortage and inventory**: Flow of goods +inventory-shortage must meet customer demand (basically, demand minus inventory must be satisfied by flow and shortage).
$$
\sum_{i: (i,c) \in A} x_{ic} + s_c = q_c - I^C_c \quad \forall c \in C
$$

**Depot capacity (except Birmingham)**: Flow of each depot must respect its effective capacity (reliable capacity) and inventory.
$$
\sum_{j: (d,j) \in A} x_{dj} \le T_d  y_d+ I^D_d \, \quad \forall d \in D \setminus \{\text{Birmingham}\}
$$

**Birmingham capacity with expansion**:Flow of each depot must respect its effective capacity (reliable capacity) and inventory. But this is the only place that has the capability of increasing capacity
$$
\sum_{j: (\text{Birmingham}, j) \in A} x_{\text{Birmingham}, j} \le (T_{\text{Birmingham}} + \Delta z) \, y_{\text{Birmingham}}+I_{\text{Birmingham}}
$$

**Limit number of open depots**:The total number of depot cannot be larger than pre-specified number (e.g., M=4)
$$
\sum_{d \in D} y_d \le M
$$

**Forced-open depots**: Not mandatory and user can decide. Optional constraint.
$$
y_d = 1 \quad \forall d \in D_{\text{force}}
$$

**Total inventory cap**: Total amount of inventory can be held at the entire supply chan network.
$$
\sum_{f \in F} I^F_f + \sum_{d \in D} I^D_d + \sum_{c \in C} I^C_c \le I_{\max}
$$

---
## Python Implementation

We import the Gurobi Python Module and other Python libraries.

---
# Part 2 — Launch the Model
Run both cells below. The control panel appears under the second cell. **To reset to baseline at any time, re-run the second cell.**

In [ ]:
%pip install -q gurobipy

In [ ]:
# ==== Interactive Supply Network Experiments (Gurobi) ====
# Single-period network design with per-site inventory holding costs.
# Decides factory/depot opening/expansion, flows, shortages, and where to hold inventory.

#%pip install gurobipy

import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display, clear_output
import ipywidgets as w

import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# FIXED TRANSPORTATION ARCS & COSTS (do not edit in class)
# -----------------------------
arcs, cost = gp.multidict({
    ('Liverpool', 'Newcastle'): 0.5,
    ('Liverpool', 'Birmingham'): 0.5,
    ('Liverpool', 'London'): 1.0,
    ('Liverpool', 'Exeter'): 0.2,
    ('Liverpool', 'Bristol'): 0.6,
    ('Liverpool', 'Northampton'): 0.4,
    ('Liverpool', 'C1'): 1.0,
    ('Liverpool', 'C3'): 1.5,
    ('Liverpool', 'C4'): 2.0,
    ('Liverpool', 'C6'): 1.0,
    ('Brighton', 'Birmingham'): 0.3,
    ('Brighton', 'London'): 0.5,
    ('Brighton', 'Exeter'): 0.2,
    ('Brighton', 'Bristol'): 0.4,
    ('Brighton', 'Northampton'): 0.3,
    ('Brighton', 'C1'): 2.0,
    ('Newcastle', 'C2'): 1.5,
    ('Newcastle', 'C3'): 0.5,
    ('Newcastle', 'C5'): 1.5,
    ('Newcastle', 'C6'): 1.0,
    ('Birmingham', 'C1'): 1.0,
    ('Birmingham', 'C2'): 0.5,
    ('Birmingham', 'C3'): 0.5,
    ('Birmingham', 'C4'): 1.0,
    ('Birmingham', 'C5'): 0.5,
    ('London', 'C2'): 1.5,
    ('London', 'C3'): 2.0,
    ('London', 'C5'): 0.5,
    ('London', 'C6'): 1.5,
    ('Exeter', 'C3'): 0.2,
    ('Exeter', 'C4'): 1.5,
    ('Exeter', 'C5'): 0.5,
    ('Exeter', 'C6'): 1.5,
    ('Bristol', 'C1'): 1.2,
    ('Bristol', 'C2'): 0.6,
    ('Bristol', 'C3'): 0.5,
    ('Bristol', 'C5'): 0.3,
    ('Bristol', 'C6'): 0.8,
    ('Northampton', 'C2'): 0.4,
    ('Northampton', 'C4'): 0.5,
    ('Northampton', 'C5'): 0.6,
    ('Northampton', 'C6'): 0.9
})

# -----------------------------
# DEFAULT (EDITABLE) DATA
# -----------------------------
supply0 = {'Liverpool': 150000, 'Brighton': 200000}
through0 = {
    'Newcastle': 70000, 'Birmingham': 50000, 'London': 100000,
    'Exeter': 40000, 'Bristol': 30000, 'Northampton': 25000
}
opencost0 = {'Newcastle': 10000, 'Birmingham': 0, 'London': 0, 'Exeter': 5000, 'Bristol': 12000, 'Northampton': 4000}
demand0 = {'C1': 50000, 'C2': 10000, 'C3': 40000, 'C4': 35000, 'C5': 60000, 'C6': 20000}
shortcost0 = {'C1': 5, 'C2': 5, 'C3': 5, 'C4': 5, 'C5': 5, 'C6': 5}
factory_reliab0 = {'Liverpool': 0.95, 'Brighton': 0.90}
depot_reliab0   = {'Newcastle': 0.92, 'Birmingham': 0.98, 'London': 0.97, 'Exeter': 0.90, 'Bristol': 0.88, 'Northampton': 0.90}

# Per-unit holding costs (customer finished goods are most expensive)
hold_factory0  = {'Liverpool': 0.03, 'Brighton': 0.03}
hold_depot0    = {'Newcastle': 0.05, 'Birmingham': 0.05, 'London': 0.05,
                  'Exeter': 0.05, 'Bristol': 0.05, 'Northampton': 0.05}
hold_customer0 = {'C1': 0.10, 'C2': 0.10, 'C3': 0.10, 'C4': 0.10, 'C5': 0.10, 'C6': 0.10}

# Network-wide inventory CAP (maximum ending inventory, units)
total_inventory_target0 = 20000

# Global sets inferred from fixed arcs
factories = sorted({i for i,j in arcs if i not in {'Newcastle','Birmingham','London','Exeter','Bristol','Northampton'}})
depots    = sorted({'Newcastle','Birmingham','London','Exeter','Bristol','Northampton'})
customers = sorted({j for i,j in arcs if str(j).startswith('C')})

# -----------------------------
# Helper: build one row of widgets for each dict item
# -----------------------------
def make_number_grid(d: dict, is_int=True, minv=None, maxv=None):
    rows = {}
    for k, v in d.items():
        if is_int:
            wgt = w.IntText(value=int(v), description=str(k), layout=w.Layout(width='300px'))
            if minv is not None: wgt.min = minv
        else:
            # For reliabilities, use sliders 0..1
            if maxv == 1 and minv == 0:
                wgt = w.FloatSlider(value=float(v), min=0, max=1, step=0.01, description=str(k),
                                    readout_format='.2f', layout=w.Layout(width='400px'))
            else:
                wgt = w.FloatText(value=float(v), description=str(k), layout=w.Layout(width='300px'))
        rows[k] = wgt
    box = w.VBox(list(rows.values()))
    return rows, box

# Widgets for each data block
sup_rows,  sup_box   = make_number_grid(supply0, is_int=True, minv=0)
thr_rows,  thr_box   = make_number_grid(through0, is_int=True, minv=0)
open_rows, open_box  = make_number_grid(opencost0, is_int=True, minv=0)
dem_rows,  dem_box   = make_number_grid(demand0, is_int=True, minv=0)
sho_rows,  sho_box   = make_number_grid(shortcost0, is_int=True, minv=0)
frel_rows, frel_box  = make_number_grid(factory_reliab0, is_int=False, minv=0, maxv=1)
drel_rows, drel_box  = make_number_grid(depot_reliab0,   is_int=False, minv=0, maxv=1)

# Holding cost widgets + total inventory target
hf_rows, hf_box = make_number_grid(hold_factory0,  is_int=False)
hd_rows, hd_box = make_number_grid(hold_depot0,    is_int=False)
hc_rows, hc_box = make_number_grid(hold_customer0, is_int=False)
inv_target = w.IntText(value=total_inventory_target0, description='Total inv cap')

# Policy/structural controls
forced_open = w.SelectMultiple(
    options=depots, value=(),
    description='Force-open depots', layout=w.Layout(width='320px', height='140px')
)
depot_limit = w.IntSlider(value=4, min=1, max=len(depots), step=1, description='Max open depots')

expand_extra_cap = w.IntText(value=20000, description='Bham expansion +cap')
expand_cost = w.IntText(value=3000, description='Bham expand cost')
obj_shift_note = w.HTML("<i>(Objective constant shift kept; it doesn’t affect optimality.)</i>")

solve_btn = w.Button(description='Solve with Gurobi', button_style='success')
out = w.Output()

accordion = w.Accordion(children=[
    sup_box, frel_box,
    thr_box, drel_box,
    open_box,
    dem_box, sho_box,
    w.VBox([forced_open, depot_limit, expand_extra_cap, expand_cost, obj_shift_note]),
    w.VBox([hf_box, hd_box, hc_box, inv_target])
])
accordion.set_title(0, 'Factory Supply')
accordion.set_title(1, 'Factory Reliability (0–1)')
accordion.set_title(2, 'Depot Throughput')
accordion.set_title(3, 'Depot Reliability (0–1)')
accordion.set_title(4, 'Depot Opening Cost')
accordion.set_title(5, 'Customer Demand')
accordion.set_title(6, 'Shortage Cost per Unit')
accordion.set_title(7, 'Policy: forced open, depot limit, Birmingham expansion')
accordion.set_title(8, 'Inventory: holding costs + CAP')

# Visualization toggle
show_network = w.Checkbox(value=True, description='Show network diagram')
accordion.children = tuple(list(accordion.children) + [w.VBox([show_network])])
accordion.set_title(len(accordion.children)-1, 'Visualization')

display(accordion, solve_btn, out)

# -----------------------------
# Build & solve model from widget values
# -----------------------------

def run_model():
    # Collect values
    supply      = {k: int(v.value)   for k,v in sup_rows.items()}
    through     = {k: int(v.value)   for k,v in thr_rows.items()}
    opencost    = {k: int(v.value)   for k,v in open_rows.items()}
    demand      = {k: int(v.value)   for k,v in dem_rows.items()}
    shortcost   = {k: int(v.value)   for k,v in sho_rows.items()}
    factory_rel = {k: float(v.value) for k,v in frel_rows.items()}
    depot_rel   = {k: float(v.value) for k,v in drel_rows.items()}
    forced      = list(forced_open.value)
    max_open    = int(depot_limit.value)
    extra_cap   = int(expand_extra_cap.value)
    exp_cost    = int(expand_cost.value)

    # Holding costs and inventory CAP
    holdF = {k: float(v.value) for k, v in hf_rows.items()}
    holdD = {k: float(v.value) for k, v in hd_rows.items()}
    holdC = {k: float(v.value) for k, v in hc_rows.items()}
    total_inventory_cap = int(inv_target.value)

    # Effective capacities by reliability
    eff_supply  = {f: supply[f] * factory_rel[f] for f in supply}
    eff_through = {d: through[d] * depot_rel[d] for d in through}

    m = gp.Model('SupplyNetworkDesign_Interactive')

    # Variables
    flow     = m.addVars(arcs, obj=cost, name="flow")
    openvar  = m.addVars(depots, obj=opencost, vtype=GRB.BINARY, name="open")
    expand   = m.addVar(obj=exp_cost, vtype=GRB.BINARY, name="expand")
    shortage = m.addVars(customers, obj=shortcost, name="shortage")

    # Ending inventory at each tier
    invF = m.addVars(factories, name="invF", lb=0.0)
    invD = m.addVars(depots,    name="invD", lb=0.0)
    invC = m.addVars(customers, name="invC", lb=0.0)

    obj = (
        gp.quicksum(cost[i, j] * flow[i, j] for i, j in arcs) +
        gp.quicksum(opencost[d] * openvar[d] for d in depots) +
        exp_cost * expand +
        gp.quicksum(shortcost[c] * shortage[c] for c in customers) +
        gp.quicksum(holdF[f] * invF[f] for f in factories) +
        gp.quicksum(holdD[d] * invD[d] for d in depots) +
        gp.quicksum(holdC[c] * invC[c] for c in customers)
    )
    m.setObjective(obj, GRB.MINIMIZE)
    # Force-open chosen depots
    for d in forced:
        openvar[d].lb = 1

    # Objective constant shift (doesn't affect optimality)
    #m.objcon = -(opencost.get('Newcastle',0) + opencost.get('Exeter',0))

    # Add holding costs to the objective (on top of current objective terms)
    #m.setObjective(
   #     m.getObjective()
      #  + gp.quicksum(holdF[f] * invF[f] for f in factories)
     #   + gp.quicksum(holdD[d] * invD[d] for d in depots)
     #   + gp.quicksum(holdC[c] * invC[c] for c in customers),
     #   GRB.MINIMIZE
    #)

    # Constraints
    # Factories: outbound + ending_inv <= effective supply
    m.addConstrs((
        gp.quicksum(flow.select(f, '*'))  <= eff_supply[f]+invF[f]
        for f in factories
    ), name="factory_balance_with_inv")

    # Depots: inbound = outbound + ending_inv
    m.addConstrs((
        gp.quicksum(flow.select('*', d)) == gp.quicksum(flow.select(d, '*'))
        for d in depots
    ), name="depot_balance_with_inv")

    # Customers: inbound + shortage = demand + ending_inv  (correct sign)
    m.addConstrs((
        gp.quicksum(flow.select('*', c)) + shortage[c] == demand[c] - invC[c]
        for c in customers
    ), name="demand_with_inv")

    # Depot capacity tying to open decisions (Birmingham may expand)
    for d in depots:
        if d == 'Birmingham':
            continue
        m.addConstr(gp.quicksum(flow.select(d, '*')) <= eff_through[d] * openvar[d]+ invD[d], name=f"cap_{d}")

    # Birmingham: limit OUTBOUND (fixed bug)
    m.addConstr(
        gp.quicksum(flow.select('Birmingham', '*')) <= (eff_through['Birmingham'] + extra_cap * expand) * openvar['Birmingham']+ invD['Birmingham'],
        name="bham_cap"
    )

    # Limit how many depots can be open
    m.addConstr(openvar.sum() <= max_open, name="open_limit")

    # Network-wide inventory CAP (maximum)
    m.addConstr(
        gp.quicksum(invF.values()) + gp.quicksum(invD.values()) + gp.quicksum(invC.values()) <= total_inventory_cap,
        name="total_inventory_cap"
    )

    # Optimize
    m.optimize()

    result = {
        "status": m.Status,
        "objective": m.ObjVal if m.Status == GRB.OPTIMAL else None,
        "open_depots": [d for d in depots if m.Status == GRB.OPTIMAL and openvar[d].X > 0.5],
        "expand_birmingham": (m.Status == GRB.OPTIMAL and round(expand.X) == 1),
    }

    if m.Status == GRB.OPTIMAL:
        # Shortages
        sh = [(c, shortage[c].X) for c in customers if shortage[c].X > 1e-6]
        shortage_df = pd.DataFrame(sh, columns=['Customer','Shortage'])

        # Flows
        flows = [(i,j, flow[i,j].X) for (i,j) in arcs if flow[i,j].X > 1e-6]
        flow_df = pd.DataFrame(flows, columns=['From','To','Quantity']).sort_values(['From','To'])

        # Service levels per customer (Satisfied = Demand - Shortage - EndingInv)
        svc_rows = []
        tot_demand = 0.0
        tot_satisfied = 0.0
        for c in customers:
            dem = float(demand[c])
            shrt = float(shortage[c].X)
            endinv = float(invC[c].X)
            sat = max(dem - shrt - endinv, 0.0)
            sl = 1.0 - (shrt / dem) if dem > 0 else 1.0
            svc_rows.append((c, dem, sat, shrt, endinv, sl))
            tot_demand += dem
            tot_satisfied += sat
        service_df = pd.DataFrame(svc_rows, columns=['Customer','Demand','Satisfied','Shortage','EndInv','ServiceLevel'])
        overall_service_level = (tot_satisfied / tot_demand) if tot_demand > 0 else 1.0

        # Inventory report
        invF_df = pd.DataFrame([(f, invF[f].X, holdF[f]) for f in factories],
                               columns=['Location','EndInv','HoldCost'])
        invD_df = pd.DataFrame([(d, invD[d].X, holdD[d]) for d in depots],
                               columns=['Location','EndInv','HoldCost'])
        invC_df = pd.DataFrame([(c, invC[c].X, holdC[c]) for c in customers],
                               columns=['Location','EndInv','HoldCost'])

        invF_df['Tier'] = 'Factory'
        invD_df['Tier'] = 'Depot'
        invC_df['Tier'] = 'Customer'
        inventory_df = pd.concat([invF_df, invD_df, invC_df], ignore_index=True)
        total_end_inv = float(inventory_df['EndInv'].sum())

        return result, shortage_df, flow_df, service_df, overall_service_level, inventory_df, total_end_inv
    else:
        return result, None, None, None, None, None, None

def draw_network_diagram(flow_df, open_depots):
    """
    Draw a 3-tier supply chain diagram using NetworkX.
    - flow_df columns: From, To, Quantity (positive flows only)
    - open_depots: list of depot names that are open in the solution
    """
    if flow_df is None or flow_df.empty:
        print("No positive flows to plot.")
        return

    # --- Build tiers from fixed sets ---
    tier_lists = [factories, depots, customers]

    # --- Fixed layout positions ---
    pos = {}
    x_spacing = 3.0
    y_spacing = 1.5
    for col, tier_nodes in enumerate(tier_lists):
        for row, node in enumerate(tier_nodes):
            pos[node] = (col * x_spacing, -row * y_spacing)

    # --- Build graph ---
    G = nx.DiGraph()
    G.add_nodes_from(factories)
    G.add_nodes_from(depots)
    G.add_nodes_from(customers)

    # Prepare a product_flow DataFrame (rename Quantity→Flow)
    product_flow = flow_df.rename(columns={'Quantity': 'Flow'})[['From', 'To', 'Flow']]

    # Sanity check for missing nodes
    all_nodes = set(product_flow["From"]).union(product_flow["To"])
    defined_nodes = set(factories + depots + customers)
    missing = all_nodes - defined_nodes
    if missing:
        print("⚠️ Warning: Unassigned nodes in tiers:", missing)

    # Add edges
    for _, row in product_flow.iterrows():
        u, v, q = row["From"], row["To"], float(row["Flow"])
        G.add_edge(u, v, weight=q)

    # --- Visual styling ---
    node_colors = []
    for n in G.nodes():
        if n in factories:
            node_colors.append("#8ecae6")      # factories
        elif n in customers:
            node_colors.append("#f4a261")      # customers
        else:
            node_colors.append("#90be6d" if n in open_depots else "#adb5bd")  # depots

    # Edge width scaling by flow
    flows = [d['weight'] for _,_,d in G.edges(data=True)]
    fmin, fmax = (min(flows), max(flows)) if flows else (0, 1)
    def width_scale(x, lo=0.8, hi=6.0):
        if fmax == fmin:
            return (lo + hi) / 2.0
        return lo + (x - fmin) * (hi - lo) / (fmax - fmin)
    edge_widths = [width_scale(d['weight']) for _,_,d in G.edges(data=True)]

    # --- Draw ---
    plt.figure(figsize=(12, 6))
    nx.draw_networkx_nodes(G, pos, node_size=1000, node_color=node_colors, edgecolors="black")
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=16, width=edge_widths)

    # Edge labels (flows, rounded)
    edge_labels = {(u, v): f"{d['weight']:.0f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=8)

    plt.title("3-Tier Supply Chain Network (positive flows)")
    plt.axis("off")
    plt.show()

@out.capture(clear_output=True)
def on_solve_clicked(b):
    try:
        res, shortage_df, flow_df, service_df, overall_sl, inventory_df, total_end_inv = run_model()
        if res["status"] == GRB.OPTIMAL:
            print(f"Optimal objective: {res['objective']:.2f}")
            print("Open depots:", res['open_depots'])
            print("Expand Birmingham?:", res['expand_birmingham'])

            # Service levels
            if service_df is not None and not service_df.empty:
                display(
                    service_df
                    .assign(ServiceLevelPct=lambda df: df['ServiceLevel']*100)
                    .style
                    .format({'Demand':'{:.0f}','Satisfied':'{:.0f}','Shortage':'{:.0f}','EndInv':'{:.0f}','ServiceLevel':'{:.2f}','ServiceLevelPct':'{:.2f}%'})
                    .set_caption("Customer Service Levels = Satisfied =1-  (Shortage/Demand)")
                )
                #print(f"Overall weighted service level: {overall_sl*100:.2f}%")
            else:
                print("No customers found.")

            # Shortages (nonzero only)
            if shortage_df is not None and not shortage_df.empty:
                display(shortage_df.style.format({'Shortage':'{:.0f}'}).set_caption("Shortages (nonzero)"))
            else:
                print("Shortages: none")

            # Positive flows
            if flow_df is not None and not flow_df.empty:
                display(flow_df.style.format({'Quantity':'{:.0f}'}).set_caption("Positive flows"))
            else:
                print("No positive flows (unexpected).")

            # Inventory placement
            if inventory_df is not None and not inventory_df.empty:
                display(
                    inventory_df
                    .sort_values(['Tier','Location'])
                    .style
                    .format({'EndInv':'{:.0f}','HoldCost':'{:.2f}'})
                    .set_caption(f"Ending Inventory by Location (Total = {total_end_inv:.0f})")
                )
            else:
                print("No ending inventory.")

            if show_network.value:
                draw_network_diagram(flow_df, res['open_depots'])
        else:
            print("Model did not reach OPTIMAL status. Status code:", res["status"])
    except gp.GurobiError as e:
        print("Gurobi error:", e)
    except Exception as e:
        print("Error:", e)

solve_btn.on_click(on_solve_clicked)
print("Ready. Expand the panels, tweak values, and click 'Solve with Gurobi'.")

---
# Part 3 — Round 1: Baseline Diagnosis 🎯 (15 pts)
*Suggested time: 20 minutes. Do not change any inputs. Click **Solve with Gurobi** once and answer from that output.*

Useful definition: **effective capacity = nominal capacity × reliability.**

**Q1.1** (2 pts) What is the optimal total network cost?

> ✏️ **Your answer:**
>
> 

**Q1.2** (2 pts) Which depots actually carry product? Is any depot listed as *open* while carrying zero flow? If so, explain why the solver would allow that. *(Hint: look at the opening costs.)*

> ✏️ **Your answer:**
>
> 

**Q1.3** (3 pts) Does the model expand Birmingham? Using Birmingham's **effective** capacity and its actual outbound flow, show numerically why this decision makes sense.

> ✏️ **Your answer:**
>
> 

**Q1.4** (4 pts) Break the total cost into its five components: **transport, depot opening, expansion, shortage, and holding.** Show how you calculated each one. Your components must add up to your answer in 1.1.

> ✏️ **Your answer:**
>
> 

**Q1.5** (4 pts) Compute the effective capacity and utilization (%) of **each factory** and **each depot that carries flow**. Where is the bottleneck in this network, factories or depots? What does that tell management about where to invest first?

> ✏️ **Your answer:**
>
> 

---
# Part 4 — Round 2: Stress Tests 🎯 (20 pts)
*Suggested time: 35 minutes. **Reset to baseline before every question.** Record each run in the Scenario Log.*

For every question report: (a) the new total cost, (b) the change versus baseline in $ and %, and (c) a short business explanation of *why* the model responded the way it did.

**Q2.1 — Depot limit** (3 pts) Set **Max open depots** to 3 and solve. Then set it to 2 and solve. Why does a limit of 3 change nothing? What does limiting the company to 2 depots cost, and which depot is sacrificed?

> ✏️ **Your answer:**
>
> 

**Q2.2 — Is the expansion worth it?** (3 pts) Set **Bham expand cost** to 20,000 so the expansion becomes unattractive. What is the new total cost? Based on this, what is the **maximum** the company should be willing to pay for the 20,000-ton Birmingham expansion?

> ✏️ **Your answer:**
>
> 

**Q2.3 — Depot outage** (3 pts) A fire shuts Birmingham. Set Birmingham's **depot reliability** to 0.00. What happens to cost, and which depot absorbs the lost volume? Look closely at Birmingham's flow in the results: is anything surprising? What does that reveal about how the expansion is modeled?

> ✏️ **Your answer:**
>
> 

**Q2.4 — Key-account surge** (2 pts) Customer C5 wins a big contract. Raise **C5 demand** from 60,000 to 78,000 (+30%). What is the cost increase and the cost **per extra ton**? How does the network adapt?

> ✏️ **Your answer:**
>
> 

**Q2.5 — What is a shortage worth?** (4 pts) Set the **shortage cost** for all customers to 0 and solve. Then set it to 1, then to 2. Describe what happens in each case. What is the break-even shortage penalty, and what does that number represent economically? Why must a planner choose this parameter carefully?

> ✏️ **Your answer:**
>
> 

**Q2.6 — Factory strike** (3 pts) Brighton workers strike. Set Brighton's **factory reliability** to 0.20. Report the cost, total shortage, and each customer's service level. Who *decided* which customers were shorted, and would a real account manager accept that choice?

> ✏️ **Your answer:**
>
> 

**Q2.7 — Political depot** (2 pts) A regional politician pushes the company to keep Newcastle open. **Force-open Newcastle** and solve. What does this cost, and does Newcastle ship anything? Write the two-sentence reply you would send to the politician.

> ✏️ **Your answer:**
>
> 

---
# Part 5 — Round 3: Model Audit 🔍 (10 pts + 3 bonus)
*Suggested time: 20 minutes.* Every model is wrong; the useful question is **where** and **how much it matters**. A senior analyst never presents a result they have not stress-tested for logic. Compare the math in the Briefing Book, the comments in the code, and the code itself.

**Q3.1** (3 pts) In the **baseline** service-level table, look at customer C4 (and C6). *Satisfied* is less than *Demand*, yet the service level shows 100%. Using the customer-demand constraint, explain how that can happen. Is this service metric telling management the truth?

> ✏️ **Your answer:**
>
> 

**Q3.2** (4 pts) Set **Total inv cap** to 215,000 (equal to total demand) and solve. Report the total cost, how much each factory produces, and which depots carry flow. Explain in plain business language what the model is doing and why a real company could not do this.

> ✏️ **Your answer:**
>
> 

**Q3.3** (3 pts) Propose a fix. Rewrite the customer-demand constraint, and any other inventory constraint you believe is flawed, so that holding inventory has a realistic meaning. Give it in words **and** in math. *(Hint: read the code comments next to each constraint and compare them with the code.)*

> ✏️ **Your answer:**
>
> 

**Q3.4** (Bonus +3) Can a **closed** depot ship product in this model? Design an experiment that proves your answer. *(Hint: make customer holding cost very expensive, e.g., 5.0 for every customer, and inspect which depots are open versus which ship.)* Identify the exact constraint responsible.

> ✏️ **Your answer:**
>
> 

---
# Part 6 — Round 4: Innovation Challenge 💡 (35 pts + 3 bonus)
Your team receives **one mandate card** from the instructor (or draws one). There is no single right answer. You are graded on the quality of your evidence, your reasoning about trade-offs, and your creativity in going beyond what the model can do.

Start every mandate from the baseline. Board policy locks (inventory cap 20,000, shortage cost $5) apply unless your card says otherwise.

## 🃏 Mandate Cards

**Card A — The Cost Cutter.** The CFO wants total network cost at or below **$166,600** with 100% service. A vendor offers three deals; you may accept **at most two**. Not every deal is a good deal.
1. *Birmingham mega-expansion:* +40,000 tons for $9,000 (replaces the standard 20,000-ton option).
2. *Bristol lease renegotiation:* opening cost drops from $12,000 to $4,000.
3. *Northampton upgrade:* throughput rises from 25,000 to 40,000 tons, but opening cost rises to $10,000.

**Card B — The Resilience Officer.** The board fears two disruptions: a **Birmingham depot fire** (reliability 0.00) and a **Brighton strike** (factory reliability 0.20). Quantify the cost and service damage of each. Propose a pre-emptive strategy and show what it costs in *normal* times (the insurance premium) versus what it protects when disaster strikes. Consider how likely each event is.

**Card C — The Growth Planner.** Demand grows **20% for every customer** next year. Which depots and which expansion should the company commit to now? Compare total cost and cost per ton with today. Where is the next bottleneck, and at what growth level would you need new capacity?

**Card D — The Key Account Guardian.** C5, the largest customer, will leave unless (i) its contract shortage penalty is **$20/ton** and (ii) it is served from **at least two different depots** (dual sourcing) so no single depot failure can cut it off. The control panel cannot enforce (ii) directly. Find a way, with parameters or with code, and report what it costs to keep C5.

**Card E — The Lean Network.** The COO wants to run with **no more than 2 depots**. Finance also reveals that every open depot carries **$15,000/year of overhead** that is not in the model (add it to every depot's opening cost, including Birmingham and London). Is "lean" worth it? At what overhead level would a 2-depot network become the better choice?

**Card F — The Green Mandate.** Brighton runs on renewable power; Liverpool does not. The sustainability office proposes capping **Liverpool production at 50,000 tons** (remember: effective supply = nominal × reliability). What does this cost per ton of production shifted? Recommend whether to adopt it, and propose one additional green lever the model does not capture.

## ✅ Every team must deliver

1. **Recommendation:** open depots, expansion decision, key flows, total cost versus baseline, and service levels.
2. **Evidence:** at least **3 model runs** in the Scenario Log that support the recommendation, including at least one run that *challenges* it.
3. **Trade-off:** what you give up and why it is worth it.
4. **Beyond the Model:** one important factor the model cannot capture (e.g., lead time, carbon, labor risk, customer priority, multi-period demand). Explain how you would add it: what new data, decision variable, or constraint.
5. **Risks and assumptions:** what would make your recommendation wrong?

**Bonus (+3):** actually implement your Beyond-the-Model idea in code (new constraint, cost term, or report) and show its effect.

### Mandate analysis
*Use the cells below for your analysis. Add as many code or markdown cells as you need.*

> ✏️ **Our mandate and recommendation:**
>
> 

In [ ]:
# Optional: code extensions for your mandate (bonus). Copy the model cell and modify it here.

## 📒 Scenario Log
*Add a row for every run. Baseline is filled in for you after you complete Q1.1.*

| Run # | Question / purpose | What you changed (vs. baseline) | Total cost | Δ vs. baseline | Depots carrying flow | Expand Bham? | Total shortage | Key insight |
|---|---|---|---|---|---|---|---|---|
| 0 | Baseline | nothing | | — | | | | |
| 1 | | | | | | | | |
| 2 | | | | | | | | |
| 3 | | | | | | | | |
| 4 | | | | | | | | |
| 5 | | | | | | | | |

---
# Part 7 — Round 5: Board Presentation 🎤 (20 pts)

**Format:** 8 minutes of presentation + 4 minutes of Board Q&A. Every team member speaks.

**Suggested deck (6–8 slides):**
1. Mandate and **recommendation up front** (the answer first, not last)
2. Baseline versus recommended network (cost, service, depots)
3. Network diagram of your recommendation
4. Evidence: the key scenario runs
5. Trade-offs, risks, and assumptions
6. Beyond the Model: your innovation
7. Next steps and what you are asking the board to approve

**⚡ The Board Twist.** During Q&A the instructor will reveal a surprise event (for example, a disruption or a demand shift). You get 60 seconds to explain how your recommendation holds up and what you would change. You cannot prepare the answer, but you can prepare by understanding your network's weak points.

**Submit** (one per team): this completed notebook (with outputs) and your slide deck.

---
# 📊 Grading Rubric (summary)

| Component | Points | Exemplary looks like |
|---|---|---|
| Round 1: Baseline Diagnosis | 15 | Correct numbers, calculations shown, components reconcile to the total, bottleneck correctly identified |
| Round 2: Stress Tests | 20 | Correct cost and Δ for each test, and each result is explained with the *mechanism* (which constraint or cost drove it) |
| Round 3: Model Audit | 10 | Flaw clearly explained in business terms, experiment reported, fix stated in words and valid math |
| Round 4: Evidence and analysis | 12 | ≥3 well-chosen runs, including one that challenges the recommendation; numbers are correct and comparable |
| Round 4: Recommendation and trade-offs | 11 | Clear, decision-ready recommendation; explicit trade-offs; risks and assumptions named |
| Round 4: Innovation beyond the model | 12 | Original, relevant idea with a concrete plan for how to model it (data, variable, constraint) |
| Presentation: communication and visuals | 12 | Answer-first storyline, clean visuals, on time, every member contributes |
| Presentation: Q&A and Board Twist | 8 | Confident, evidence-based answers; adapts the recommendation to the twist in real time |
| **Total** | **100** | |
| Bonus: Q3.4 closed-depot proof | +3 | |
| Bonus: working code extension | +3 | |

*Individual grades may be adjusted based on peer evaluation.*